In [4]:
import numpy as np
import os

# 欠陥ブロック生成関数（12x12用）
def generate_defect_vertices(start_vertex, num_blocks):
    """
    各ブロックの開始頂点番号を生成する関数。
    
    Parameters:
    start_vertex (int): 最初の頂点番号（例：32）。
    num_blocks (int): 生成するブロックの数（例：90）。
    
    Returns:
    list: 各ブロックの開始頂点番号のリスト。
    """
    defect_vertices = []

    # 各ブロックの開始頂点番号を生成
    for block in range(num_blocks):
        defect_vertices.append(start_vertex + block * num_rows * 12)

    return defect_vertices

# 既存の欠陥ブロック生成関数
def generate_defect_vertices(start_vertex, num_blocks, num_rows=29):
    """
    各ブロックの開始頂点番号を生成する関数。
    
    Parameters:
    start_vertex (int): 最初の頂点番号（例：32）。
    num_blocks (int): 生成するブロックの数（例：90）。
    
    Returns:
    list: 各ブロックの開始頂点番号のリスト。
    """
    defect_vertices = []

    # 各ブロックの開始頂点番号を生成
    for block in range(1, num_blocks + 1):
        vertices = [
            int(start_vertex),
            int(start_vertex) + num_rows * 12,
            int(start_vertex) + num_rows * 12 * 2, 
            int(start_vertex) + num_rows * 12 * 3,
            int(start_vertex) + num_rows * 12 * 4,
            int(start_vertex) + 12, 
            int(start_vertex) + num_rows * 12 + 12, 
            int(start_vertex) + num_rows * 12 * 2 + 12,
            int(start_vertex) + num_rows * 12 * 3 + 12,
            int(start_vertex) + num_rows * 12 * 4 + 12
        ]

    return vertices

def generate_blocks(start_vertices, num_rows=29):
    """
    ブロック内の頂点番号を生成する関数。
    
    Parameters:
    start_vertices (list): 各ブロックの開始頂点番号のリスト。
    num_rows (int): 縦方向の行数（デフォルトは29行）。
    
    Returns:
    blocks (list): 各ブロックに対応する頂点番号のリストを格納したリスト。
    """
    blocks = []
    
    for start in start_vertices:
        block = [
            start + row * num_rows + col
            for row in range(13)  # 縦方向に12行
            for col in range(13)  # 横方向に12列
        ]
        blocks.append(block)
    
    return blocks

# 正規化ラベルの割り当てルール
def assign_label(layer):
    label_map = {
        19: (0.95, 0.05), 18: (0.92, 0.08), 17: (0.89, 0.11), 16: (0.86, 0.14),
        15: (0.83, 0.17), 14: (0.80, 0.20), 13: (0.77, 0.23), 12: (0.74, 0.26),
        11: (0.71, 0.29), 10: (0.29, 0.71), 9: (0.26, 0.74), 8: (0.23, 0.77),
        7: (0.20, 0.80), 6: (0.17, 0.83), 5: (0.14, 0.86), 4: (0.11, 0.89),
        3: (0.08, 0.92), 2: (0.05, 0.95)
    }
    return label_map.get(layer, (0.0, 0.0))

# 各ブロックの頂点にラベルを割り当てる関数
def assign_labels_to_blocks(block1_vertices, block2_vertices, layer):
    labels = np.zeros(3654)  # 3654頂点を初期化
    high_label, low_label = assign_label(layer)

    # Block1に高ラベルを、Block2に低ラベルを割り当てる
    for vertex in block1_vertices:
        if vertex < 3654:
            labels[vertex] = high_label

    for vertex in block2_vertices:
        if vertex < 3654:
            labels[vertex] = low_label

    return labels

# 欠陥ラベルをファイルに保存する関数
def generate_and_save_defect_labels(start_vertex1, start_vertex2, num_blocks, num_layers, output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # 各ブロックの頂点を生成
    start_vertices1 = generate_defect_vertices(start_vertex1, num_blocks)
    start_vertices2 = generate_defect_vertices(start_vertex2, num_blocks)

    blocks1 = generate_blocks(start_vertices1)
    blocks2 = generate_blocks(start_vertices2)

    # 各層ごとにラベルを割り当ててファイルに保存
    for layer in range(2, num_layers + 1):  # 層は2〜19
        for i in range(num_blocks):
            labels = assign_labels_to_blocks(blocks1[i], blocks2[i], layer)
            filename = os.path.join(output_folder, f"12x12DefectLabel1_L{layer}B{i+1}.npy")
            np.save(filename, labels)

# 使用例
start_vertex1 = 32
start_vertex2 = 1859
num_blocks = 10
num_layers = 19
output_folder = "/home/nishioka/GNN/Defect_12x12_data/DefectLabels_12x12"

generate_and_save_defect_labels(start_vertex1, start_vertex2, num_blocks, num_layers, output_folder)

In [2]:
import numpy as np
import os

# 修正した欠陥ブロック生成関数
def generate_defect_vertices(start_vertex, num_blocks, num_rows=29):
    """
    各ブロックの開始頂点番号を生成する関数。

    Parameters:
    start_vertex (int): 最初の頂点番号（例：32）。
    num_blocks (int): 生成するブロックの数（例：10）。
    num_rows (int): グリッドの横方向のセル数（デフォルトは29）。

    Returns:
    list: 各ブロックの開始頂点番号のリスト。
    """
    defect_vertices = []

    # 上から順番にブロックの開始頂点を生成
    # ブロック1〜5（左側の列）
    for i in range(5):
        block_start_vertex = int(start_vertex) + num_rows * 12 * i
        defect_vertices.append(block_start_vertex)

    # ブロック6〜10（右側の列）
    for i in range(5):
        block_start_vertex = int(start_vertex) + num_rows * 12 * i + 12
        defect_vertices.append(block_start_vertex)

    return defect_vertices

# ブロック内の頂点番号を生成する関数
def generate_blocks(start_vertices, num_rows=29):
    """
    ブロック内の頂点番号を生成する関数。

    Parameters:
    start_vertices (list): 各ブロックの開始頂点番号のリスト。
    num_rows (int): 縦方向の行数（デフォルトは29行）。

    Returns:
    blocks (list): 各ブロックに対応する頂点番号のリストを格納したリスト。
    """
    blocks = []
    
    for start in start_vertices:
        block = [
            start + row * num_rows + col
            for row in range(13)  # 縦方向に12行
            for col in range(13)  # 横方向に12列
        ]
        blocks.append(block)
    
    return blocks

# 正規化ラベルの割り当てルール
def assign_label(layer):
    label_map = {
        19: (0.05, 0.95), 18: (0.08, 0.92), 17: (0.11, 0.89), 16: (0.14, 0.86),
        15: (0.17, 0.83), 14: (0.20, 0.80), 13: (0.23, 0.77), 12: (0.26, 0.74),
        11: (0.29, 0.71), 10: (0.71, 0.29), 9: (0.74, 0.26), 8: (0.77, 0.23),
        7: (0.80, 0.20), 6: (0.83, 0.17), 5: (0.86, 0.14), 4: (0.89, 0.11),
        3: (0.92, 0.08), 2: (0.95, 0.05)
    }
    return label_map.get(layer, (0.0, 0.0))

# 各ブロックの頂点にラベルを割り当てる関数
def assign_labels_to_blocks(block1_vertices, block2_vertices, layer):
    labels = np.zeros(3654)  # 3654頂点を初期化
    high_label, low_label = assign_label(layer)

    # Block1に高ラベルを、Block2に低ラベルを割り当てる
    for vertex in block1_vertices:
        if vertex < 3654:
            labels[vertex] = high_label

    for vertex in block2_vertices:
        if vertex < 3654:
            labels[vertex] = low_label

    return labels

# 欠陥ラベルをファイルに保存する関数
def generate_and_save_defect_labels(start_vertex1, start_vertex2, num_blocks, num_layers, output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # 各ブロックの頂点を生成
    start_vertices1 = generate_defect_vertices(start_vertex1, num_blocks)
    start_vertices2 = generate_defect_vertices(start_vertex2, num_blocks)

    blocks1 = generate_blocks(start_vertices1)
    blocks2 = generate_blocks(start_vertices2)

    # 各層ごとにラベルを割り当ててファイルに保存
    for layer in range(2, num_layers + 1):  # 層は2〜19
        for i in range(num_blocks):
            labels = assign_labels_to_blocks(blocks1[i], blocks2[i], layer)
            filename = os.path.join(output_folder, f"12x12DefectLabel1_L{layer}B{i+1}.npy")
            np.save(filename, labels)

# 使用例
start_vertex1 = 32
start_vertex2 = 1859
num_blocks = 10
num_layers = 19
output_folder = "/home/nishioka/GNN/Defect_12x12_data/DefectLabels_12x12test2"

generate_and_save_defect_labels(start_vertex1, start_vertex2, num_blocks, num_layers, output_folder)
